# Objective 5: Develop and evaluate a hybrid detection framework utilising the most robst features.

Have now, from objectives 2 and 4, lists of features that a) best distinguish between human and AI generated texts and b) are most robust to paraphrasing. So now need to bring this all together into a classifier in the aim that this additional information improves detection performance.

___

...but I now realise an issue with my plan. I created 2000 human / 2000 ai dataset, split 80/20 for 800 test, then paraphrased these test 3 timed for a 3200 test split which was 800 each of original / translation / heavy paraphrases. Train has already been converted into x_train_surface / syntactic / semantic for training and the test data will be unseen.

However I now realise that whilst analysis of the discriminatory abiilty of features to detect between human and AI was based on data from the training set, the analysis of which features are most robust to paraphrasing was actually on the test data, as that is the data that had been paraphrased, and these two anslysis are vital in the classifier as the entire point is that I want to be taking the features that are both the most discriminatory and the most robust.

Whilst the classifier will not have seen the test data, it will be built, in part, using analysis from the test data, which shouldn't be the case.

Therefore, for the sake of fairness, and transparency, I am now going to take that initial 3200 train set and take just 10% extra to create a 300 test set which can then be paraphrased three times, for a total of a 1200 test set split across four paraphrases.

Whilst a 10% split is lesser than I would like, I had already taken 20% off initally, and another 20% here would result in 60%/40%, losing almost half my training data. Additioanlly, with the paraphrases the test set still retains good size.

And lastly, of course, the X_train_s... data has already been created. Some of this data will now be going into the test set, so I will also need to re-run feature extraction on the training set once the 300 texts have been removed to ensure a completely distinct training and test set.

Steps:

- get 3200 train set and split to 2900 / 300 with that 300 equal parts human and AI generated
- run paraphrase transformations on new test set for a 300/300/300/300 = 1200 test set
- re-run feature extraction pipelines on new 2900 train set for new x_train_surface / syntactic / semantic
- run 12 feature extractions on  all tests sets - original, light, heavy and trans paraphrase for surface/syntactic and semantic
- then proceed with objective 5 using new train and test sets.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

____

### Skip forward from here - splitting old 3200 train set

___

In [2]:
# lets load up existing train dataset
df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

In [4]:
print(df.shape)
print(df["author"].value_counts())

(3199, 7)
author
AI       1600
human    1599
Name: count, dtype: int64


This is as expected, there was one document removed earlier due to being Welsh language. Now want to split 300 test away that is 50% each human and AI.

In [9]:
# perform train test split to extract 300 test, split by label
_, test_df = train_test_split(
    df,
    test_size = 300,
    stratify = df["label"],
    random_state = 1985
)

# drop test to create a new train df
train_df = df.drop(index = test_df.index).copy()

# print stats
print("Train shape: ", train_df.shape)
print("Train class count: ")
print(train_df["label"].value_counts())
print("Test shape: ", test_df.shape)
print("Test class count: ")
print(test_df["label"].value_counts())

# save new train / test csvs
train_df.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv", 
    index = False
)
test_df.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_test_300.csv", 
    index = False
)

Train shape:  (2899, 7)
Train class count: 
label
1    1450
0    1449
Name: count, dtype: int64
Test shape:  (300, 7)
Test class count: 
label
1    150
0    150
Name: count, dtype: int64


Now I return to the paraphrase creation IPYNB to create paraphrases of the new 300 test set.

___

Returned here and now have a final_test_300_paraphrased csv that is the 150/150 split created above which now has light, heavy and trans based paraphrases of every text, and the new 2900 (2899) train data which has had features extracted to x_train_surface_2900, x_train_syntactic_2900 and x_train_semantic_2900.

C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_test_300_paraphrased.csv

___

## Skip forward from here - splitting test df up into individual for light, heavy etc.

Now need to take this paraphrased test set and split it into 4 different test sets, one each for original etc. and then each of these needs to go off an run through feature extraction pipelines.

In [15]:
df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_test_300_paraphrased.csv"
)

Split into four DFs and save.

In [16]:
# can drop topic etc at this stage
meta_cols = ["document_id", "label", "author"]

# create new unique dfs for each condition
test_original = df[meta_cols + ["text"]].copy()
test_light = df[meta_cols + ["light_paraphrase"]].copy()
test_trans = df[meta_cols + ["translation_paraphrase"]].copy()
test_heavy = df[meta_cols + ["heavy_paraphrase"]].copy()

# and within those dfs the text column should be called "text" as 
# this is the name of the column to be extracted and the condtition 
# will be stated in filename. Not needed for original as already text.
test_light = test_light.rename(columns = {"light_paraphrase" : "text"})
test_trans = test_trans.rename(columns = {"translation_paraphrase" : "text"})
test_heavy = test_heavy.rename(columns = {"heavy_paraphrase" : "text"})

# confirm stats
print("Original: ", test_original.shape)
print("Light: ", test_light.shape)
print("Trans: ", test_trans.shape)
print("Heavy: ", test_heavy.shape)

Original:  (300, 4)
Light:  (300, 4)
Trans:  (300, 4)
Heavy:  (300, 4)


In [17]:
# and now save all as new files to take back to feature extraction
test_original.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300.csv", 
    index = False
)
test_light.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300.csv", 
    index = False
)
test_trans.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300.csv", 
    index = False
)
test_heavy.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300.csv", 
    index = False
)

Will now return here after feature extraction on test sets.

___

# Return to executing here: now loading up files and converting into datasets for model training etc.

Now need to load up all the test sets, which encompass original, light, heavy and translation based parpahrases and for these four, surface-level, syntactic and semantic features, so 12 files.

## Loading up test DFs and labels

In [18]:
##################################
# Load up original texts
##################################
test_original_surface = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300_features_surface.csv", 
)
test_original_syntactic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300_features_syntactic.csv", 
)
test_original_semantic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300_features_semantic.csv", 
)

##################################
# Load up light paraphrased text
##################################

test_light_surface = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300_features_surface.csv", 
)
test_light_syntactic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300_features_syntactic.csv", 
)
test_light_semantic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300_features_semantic.csv", 
)

##################################
# Load up heavy paraphrased text
##################################
test_heavy_surface = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300_features_surface.csv", 
)
test_heavy_syntactic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300_features_syntactic.csv", 
)
test_heavy_semantic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300_features_semantic.csv", 
)

##################################
# Load up trans paraphrased text
##################################
test_trans_surface = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300_features_surface.csv", 
)
test_trans_syntactic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300_features_syntactic.csv", 
)
test_trans_semantic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300_features_semantic.csv", 
)

##################################
# Shape check
##################################
print("Original: ", test_original_surface.shape, test_original_syntactic.shape, test_original_semantic.shape)
print("Light: ", test_light_surface.shape, test_light_syntactic.shape, test_light_semantic.shape)
print("Heavy: ", test_heavy_surface.shape, test_heavy_syntactic.shape, test_heavy_semantic.shape)
print("Trans: ", test_trans_surface.shape, test_trans_syntactic.shape, test_trans_semantic.shape)

Original:  (300, 96) (300, 96) (300, 7)
Light:  (300, 96) (300, 96) (300, 7)
Heavy:  (300, 96) (300, 96) (300, 7)
Trans:  (300, 96) (300, 96) (300, 7)


Need to now combine these into 4 dfs - original, light etc. as main test dfs, combining features, then need to load up y test too.

In [19]:
# create tests dfs combining surface, syntactic and semantic features for each condition
X_test_original = pd.concat(
    [
        test_original_surface, 
        test_original_syntactic,
        test_original_semantic
    ],
    axis = 1
)

X_test_light = pd.concat(
    [
        test_light_surface, 
        test_light_syntactic,
        test_light_semantic
    ],
    axis = 1
)

X_test_heavy = pd.concat(
    [
        test_heavy_surface, 
        test_heavy_syntactic,
        test_heavy_semantic
    ],
    axis = 1
)

X_test_trans = pd.concat(
    [
        test_trans_surface, 
        test_trans_syntactic,
        test_trans_semantic
    ],
    axis = 1
)

print("Original: ", X_test_original.shape)
print("Light: ", X_test_light.shape)
print("Heavy: ", X_test_heavy.shape)
print("Trans: ", X_test_trans.shape)

Original:  (300, 199)
Light:  (300, 199)
Heavy:  (300, 199)
Trans:  (300, 199)


And load the y test labels.

In [20]:
df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_test_300_paraphrased.csv"
)

y_test = df["label"]
print("y_test: ", y_test.shape)
print("y_test labels: ", y_test.value_counts())

y_test:  (300,)
y_test labels:  label
1    150
0    150
Name: count, dtype: int64


___

## Loading up train DFs and labels

And now the same to load up the x train data and combine.

In [21]:
# load these up
X_train_surface = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_surface_2900.csv"
)
X_train_syntactic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_syntactic_2900.csv"
)
X_train_semantic = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_semantic_2900.csv"
)

Confirm stats.

In [22]:
print("Surface: ", X_train_surface.shape)
print("Syntactic: ", X_train_syntactic.shape)
print("Semantic: ", X_train_semantic.shape)

# print("Surface columns: ")
# print(X_train_surface.columns.tolist())
# print("Syntactic columns: ")
# print(X_train_syntactic.columns.tolist())
# print("Semantic columns: ")
# print(X_train_semantic.columns.tolist())

Surface:  (2899, 96)
Syntactic:  (2899, 96)
Semantic:  (2899, 7)


Now need to combine these 3 train feature sets into one master train set.

In [23]:
X_train_master = pd.concat(
    [
        X_train_surface,
        X_train_syntactic,
        X_train_semantic
    ],
    axis = 1
)
print(X_train_master.shape)

(2899, 199)


Also need to load up master train for y labels.

In [24]:
df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv"
)
y_train = df["label"]
print(y_train.shape)

(2899,)


___

## All datasets and labels now loaded

Task now is to specify which features will comprise the set used to create an improved (hopefully) classifier. We have features from objective 2 ranked by AUC strength that show the best at / most discriminatiory  at determining author betwene human and AI, and we have features from objective 4 which show those which are most robust, i.e. show the least degradation after paraphrasing. Bearing in mind we assessed that the first ~20/21 of these showed zero change due to the respective feature only being seen in < 50% of texts. However, they are still robust.

In [31]:
# this loads up the feature list from objective 2 which is the features best as determining human VS AI
discrim_features = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\objective_2_best_features.csv",
)
# and this one is from objective 4 which ranks the most robust
robust_features = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\objective_4_most_robust_features.csv",
)

In [32]:
print(discrim_features.shape)
discrim_features.head(50)

(199, 10)


,feature_group,feature,human_median,human_mean,ai_median,ai_mean,ROC_AUC,AUC_strength,p_value,p_value_adjusted
0,Surface,mtld,83.350515,84.876052,145.995872,150.891060,0.950266,0.950266,0.000000e+00,0.000000e+00
1,Surface,root_ttr,10.071978,9.999202,11.747654,11.721610,0.934122,0.934122,0.000000e+00,0.000000e+00
2,Surface,hapax_l_ratio,0.356098,0.353988,0.451175,0.448068,0.917411,0.917411,0.000000e+00,0.000000e+00
3,Surface,lower_ratio,0.758447,0.756193,0.796765,0.796546,0.896772,0.896772,0.000000e+00,0.000000e+00
4,Surface,alpha_ratio,0.784992,0.784359,0.811485,0.811228,0.845782,0.845782,1.907292e-251,3.662000e-250
5,Surface,upper_ratio,0.025759,0.028119,0.014174,0.014682,0.154317,0.845683,2.646726e-251,4.234762e-250
6,Syntactic,pos_bigram_NOUN_VERB,0.017621,0.019547,0.037500,0.037826,0.831029,0.831029,1.258836e-230,1.208482e-228
7,Surface,std_sentence_length,9.349233,10.570189,6.704573,6.886434,0.188690,0.811310,3.279019e-204,4.496940e-203
8,Syntactic,pos_ratio_NOUN,0.164990,0.166168,0.201784,0.217507,0.793771,0.793771,4.472663e-182,2.146878e-180
9,Surface,punct_ratio,0.030132,0.032501,0.022071,0.023880,0.226858,0.773142,1.109979e-157,1.331974e-156


In [33]:
print(robust_features.shape)
robust_features.head(50)

(184, 3)


,feature_group,feature,mean_norm_change
0,surface,func_ratio_any,0.000000
1,surface,func_ratio_will,0.000000
2,surface,func_ratio_your,0.000000
3,surface,func_ratio_every,0.000000
4,surface,func_ratio_even,0.000000
5,surface,func_ratio_down,0.000000
6,surface,func_ratio_our,0.000000
7,surface,question_ratio,0.000000
8,syntactic,dependency_ratio_discourse,0.000000
9,surface,func_ratio_my,0.000000


What we want here is a set of the 'best' features to use, but these features should score highly in both discriminatory ability and robustness. Features that discriminate well but aren't robust aren't particularly useful and features which are robust but don't discriminate well are worse.

There are 199 features in total and so I would expect to use maybe 20-30 in building the hopefully mroe accurate classifier, but we want to see the set of features that exist in the top x of both sets.

In [42]:
# define the top x features from each group
x = 70

# find intersection of top x from each group
chosen_features = (
    robust_features.head(x)
    .merge(
        discrim_features.head(x)[["feature"]],
        # by matching on feature
        on = "feature",
    )
)

final_features = chosen_features["feature"].tolist()

print("Total features: ", len(final_features))
print(chosen_features[["feature", "feature_group"]])

Total features:  20
                       feature feature_group
0   dependency_ratio_discourse     syntactic
1               pos_ratio_INTJ     syntactic
2            exclamation_ratio       surface
3               func_ratio_his       surface
4                pos_ratio_SYM     syntactic
5                func_ratio_do       surface
6        dependency_ratio_flat     syntactic
7                  digit_ratio       surface
8              pos_ratio_PROPN     syntactic
9                func_ratio_or       surface
10               pos_ratio_NUM     syntactic
11             func_ratio_than       surface
12              pos_ratio_PRON     syntactic
13                 upper_ratio       surface
14              pos_ratio_NOUN     syntactic
15              func_ratio_was       surface
16      dependency_ratio_appos     syntactic
17               pos_ratio_ADJ     syntactic
18       dependency_ratio_conj     syntactic
19        pos_bigram_VERB_PRON     syntactic


Played with x and settled on top 70 leading to an intersection of 20 best features. Went up to 100 and still no semantic make the list. 70 is already a high amount from ~200 so happy to keep it there to settle on 20 features total.

In [43]:
final_features

['dependency_ratio_discourse',
 'pos_ratio_INTJ',
 'exclamation_ratio',
 'func_ratio_his',
 'pos_ratio_SYM',
 'func_ratio_do',
 'dependency_ratio_flat',
 'digit_ratio',
 'pos_ratio_PROPN',
 'func_ratio_or',
 'pos_ratio_NUM',
 'func_ratio_than',
 'pos_ratio_PRON',
 'upper_ratio',
 'pos_ratio_NOUN',
 'func_ratio_was',
 'dependency_ratio_appos',
 'pos_ratio_ADJ',
 'dependency_ratio_conj',
 'pos_bigram_VERB_PRON']

___

Now need to use these features to create seperate x trains.

In [45]:
X_train_robust = X_train_master[final_features]
print("Full features: ", X_train_master.shape)
print("Robust features: ", X_train_robust.shape)

Full features:  (2899, 199)
Robust features:  (2899, 20)


___

And finally I now need X_test DFs that also only include the selected 'best' features.

In [48]:
X_test_original_robust = X_test_original[final_features]
X_test_light_robust = X_test_light[final_features]
X_test_heavy_robust = X_test_heavy[final_features]
X_test_trans_robust = X_test_trans[final_features]

print("Original: ", X_test_original.shape)
print("Light: ", X_test_light.shape)
print("Heavy: ", X_test_heavy.shape)
print("Trans: ", X_test_trans.shape)

print("Original robust: ", X_test_original_robust.shape)
print("Light robust: ", X_test_light_robust.shape)
print("Heavy robust: ", X_test_heavy_robust.shape)
print("Trans robust: ", X_test_trans_robust.shape)

Original:  (300, 199)
Light:  (300, 199)
Heavy:  (300, 199)
Trans:  (300, 199)
Original robust:  (300, 20)
Light robust:  (300, 20)
Heavy robust:  (300, 20)
Trans robust:  (300, 20)


___

# Step 2: Model training

Now time to train models. I use standard scaler due to the differing scales being used.

In [54]:
# logistic regression model using all features
lr_all = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter = 1000,
        random_state = 1985
    )
)

# logistic regression model using only robust features
lr_robust = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter = 1000,
        random_state = 1985
    )
)

# train all feature model
lr_all.fit(X_train_master, y_train)

# train robust feature model
lr_robust.fit(X_train_robust, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, random_state=1985))])

In [55]:
# full model on all original data
y_pred_all_original = lr_all.predict(X_test_original)

# robust model on selected robust features
y_pred_robust_original = lr_robust.predict(X_test_original_robust)

# calculate and print stats
print("All feature model - Original text:")
print("Accuracy: ", accuracy_score(y_test, y_pred_all_original))
print("Precision: ", precision_score(y_test, y_pred_all_original, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_all_original, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_all_original, zero_division = 0))

print("\nRobust features only model - Original text:")
print("Accuracy: ", accuracy_score(y_test, y_pred_robust_original))
print("Precision: ", precision_score(y_test, y_pred_robust_original, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_robust_original, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_robust_original, zero_division = 0))

All feature model - Original text:
Accuracy:  1.0
Precision:  1.0
Recall:  1.0
F1:  1.0

Robust features only model - Original text:
Accuracy:  0.9166666666666666
Precision:  0.8980891719745223
Recall:  0.94
F1:  0.9185667752442996


This 100% accuracy on full data is worrying. Will perform some checks.

In [59]:
print(X_train_master.columns.tolist())

['word_count', 'character_count', 'sentence_count', 'mean_word_length', 'std_word_length', 'mean_sentence_length', 'std_sentence_length', 'root_ttr', 'hapax_l_ratio', 'hapax_d_ratio', 'yules_k', 'mtld', 'comma_ratio', 'colon_ratio', 'semicolon_ratio', 'question_ratio', 'exclamation_ratio', 'hyphen_ratio', 'endash_ratio', 'emdash_ratio', 'alpha_ratio', 'upper_ratio', 'lower_ratio', 'digit_ratio', 'space_ratio', 'punct_ratio', 'func_ratio_a', 'func_ratio_all', 'func_ratio_also', 'func_ratio_an', 'func_ratio_and', 'func_ratio_any', 'func_ratio_are', 'func_ratio_as', 'func_ratio_at', 'func_ratio_be', 'func_ratio_been', 'func_ratio_but', 'func_ratio_by', 'func_ratio_can', 'func_ratio_do', 'func_ratio_down', 'func_ratio_even', 'func_ratio_every', 'func_ratio_for', 'func_ratio_from', 'func_ratio_had', 'func_ratio_has', 'func_ratio_have', 'func_ratio_her', 'func_ratio_his', 'func_ratio_if', 'func_ratio_in', 'func_ratio_into', 'func_ratio_is', 'func_ratio_it', 'func_ratio_its', 'func_ratio_may'

No metadata included in columns that essentially might have given the 'answer'. Lets check confusion matrix.

In [62]:
print(confusion_matrix(y_test, y_pred_all_original))

[[150   0]
 [  0 150]]


In retrospect, lots of features were covered and so it is not surprising that 100% accuracy was attained, and so similarly, it is not surprising that the model taht only contained the selected 'best' features actually performed worse as it had less information to use. Lets continue and see the stats after paraphrasing occurs to check.

___

## Testing paraphrase conditions

In [63]:
# Full model predicting on paraphrase conditions

# full model on all light paraphrase data
y_pred_all_light = lr_all.predict(X_test_light)
# full model on all heavy paraphrase data
y_pred_all_heavy = lr_all.predict(X_test_heavy)
# full model on all trans paraphrase data
y_pred_all_trans = lr_all.predict(X_test_trans)


# Robust only model predicting on paraphrase conditions

# robust model on light paraphrase robust data
y_pred_robust_light = lr_robust.predict(X_test_light_robust)
# robust model on heavy paraphrase robust data
y_pred_robust_heavy = lr_robust.predict(X_test_heavy_robust)
# robust model on trans paraphrase robust data
y_pred_robust_trans = lr_robust.predict(X_test_trans_robust)

Calculate and print stats

In [65]:
# have also bought the previous stats back in here to compare all side by side

#####################################
# Full model
#####################################

# all features model on original text
print("All feature model - Original text:")
print("Accuracy: ", accuracy_score(y_test, y_pred_all_original))
print("Precision: ", precision_score(y_test, y_pred_all_original, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_all_original, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_all_original, zero_division = 0))
print(confusion_matrix(y_test, y_pred_all_original))

# all features model on light paraphrase
print("\nAll feature model - Light paraphrase:")
print("Accuracy: ", accuracy_score(y_test, y_pred_all_light))
print("Precision: ", precision_score(y_test, y_pred_all_light, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_all_light, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_all_light, zero_division = 0))
print(confusion_matrix(y_test, y_pred_all_light))

# all features model on translation paraphrased data
print("\nAll feature model - Trans paraphrase:")
print("Accuracy: ", accuracy_score(y_test, y_pred_all_trans))
print("Precision: ", precision_score(y_test, y_pred_all_trans, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_all_trans, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_all_trans, zero_division = 0))
print(confusion_matrix(y_test, y_pred_all_trans))

# all features model on heavy paraphrased data
print("\nAll feature model - Heavy paraphrase:")
print("Accuracy: ", accuracy_score(y_test, y_pred_all_heavy))
print("Precision: ", precision_score(y_test, y_pred_all_heavy, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_all_heavy, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_all_heavy, zero_division = 0))
print(confusion_matrix(y_test, y_pred_all_heavy))

#####################################
# Robust features only model
#####################################

# robust model on original text
print("\nRobust feature model - Original text:")
print("Accuracy: ", accuracy_score(y_test, y_pred_robust_original))
print("Precision: ", precision_score(y_test, y_pred_robust_original, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_robust_original, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_robust_original, zero_division = 0))
print(confusion_matrix(y_test, y_pred_robust_original))

# Robust model on light paraphrase
print("\nRobust feature model - Light paraphrase:")
print("Accuracy: ", accuracy_score(y_test, y_pred_robust_light))
print("Precision: ", precision_score(y_test, y_pred_robust_light, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_robust_light, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_robust_light, zero_division = 0))
print(confusion_matrix(y_test, y_pred_robust_light))

# RObust model on translation paraphrased data
print("\nRobust feature model - Trans paraphrase:")
print("Accuracy: ", accuracy_score(y_test, y_pred_robust_trans))
print("Precision: ", precision_score(y_test, y_pred_robust_trans, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_robust_trans, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_robust_trans, zero_division = 0))
print(confusion_matrix(y_test, y_pred_robust_trans))

# Robust model on heavy paraphrased data
print("\nRobust feature model - Heavy paraphrase:")
print("Accuracy: ", accuracy_score(y_test, y_pred_robust_heavy))
print("Precision: ", precision_score(y_test, y_pred_robust_heavy, zero_division = 0))
print("Recall: ", recall_score(y_test, y_pred_robust_heavy, zero_division = 0))
print("F1: ", f1_score(y_test, y_pred_robust_heavy, zero_division = 0))
print(confusion_matrix(y_test, y_pred_robust_heavy))

All feature model - Original text:
Accuracy:  1.0
Precision:  1.0
Recall:  1.0
F1:  1.0
[[150   0]
 [  0 150]]

All feature model - Light paraphrase:
Accuracy:  0.97
Precision:  0.9433962264150944
Recall:  1.0
F1:  0.970873786407767
[[141   9]
 [  0 150]]

All feature model - Trans paraphrase:
Accuracy:  0.9833333333333333
Precision:  0.9738562091503268
Recall:  0.9933333333333333
F1:  0.9834983498349835
[[146   4]
 [  1 149]]

All feature model - Heavy paraphrase:
Accuracy:  0.7333333333333333
Precision:  0.6521739130434783
Recall:  1.0
F1:  0.7894736842105263
[[ 70  80]
 [  0 150]]

Robust feature model - Original text:
Accuracy:  0.9166666666666666
Precision:  0.8980891719745223
Recall:  0.94
F1:  0.9185667752442996
[[134  16]
 [  9 141]]

Robust feature model - Light paraphrase:
Accuracy:  0.89
Precision:  0.8545454545454545
Recall:  0.94
F1:  0.8952380952380953
[[126  24]
 [  9 141]]

Robust feature model - Trans paraphrase:
Accuracy:  0.8933333333333333
Precision:  0.851190476190

Lets store results for now.

In [66]:
# initialise empty list
results = []

# predictions by condition for full feature classifier
full_preds = {
    "Original" : y_pred_all_original,
    "Light" : y_pred_all_light,
    "Translation" : y_pred_all_trans,
    "Heavy" : y_pred_all_heavy
}

# same for predictions by robust classifier
robust_preds = {
    "Original" : y_pred_robust_original,
    "Light" : y_pred_robust_light,
    "Translation" : y_pred_robust_trans,
    "Heavy" : y_pred_robust_heavy
}

# easier to recalculate and store rather than manually enter
# re-calculate all stats for the full features model
for condition, predictions in full_preds.items():

    results.append({
        "Model" : "All features",
        "Condition" : condition,
        "Accuracy" : accuracy_score(y_test, predictions),
        "Precision" : precision_score(y_test, predictions, zero_division = 0),
        "Recall" : recall_score(y_test, predictions, zero_division = 0),
        "F1" : f1_score(y_test, predictions, zero_division = 0)
    })
    
# and now do the same for the robust features only model
for condition, predictions in robust_preds.items():

    results.append({
        "Model" : "Robust features",
        "Condition" : condition,
        "Accuracy" : accuracy_score(y_test, predictions),
        "Precision" : precision_score(y_test, predictions, zero_division = 0),
        "Recall" : recall_score(y_test, predictions, zero_division = 0),
        "F1" : f1_score(y_test, predictions, zero_division = 0)
    })

# store
obj_5_results_df = pd.DataFrame(results)
# print
obj_5_results_df

,Model,Condition,Accuracy,Precision,Recall,F1
0,All features,Original,1.000000,1.000000,1.000000,1.000000
1,All features,Light,0.970000,0.943396,1.000000,0.970874
2,All features,Translation,0.983333,0.973856,0.993333,0.983498
3,All features,Heavy,0.733333,0.652174,1.000000,0.789474
4,Robust features,Original,0.916667,0.898089,0.940000,0.918567
5,Robust features,Light,0.890000,0.854545,0.940000,0.895238
6,Robust features,Translation,0.893333,0.851190,0.953333,0.899371
7,Robust features,Heavy,0.790000,0.720812,0.946667,0.818444


In [67]:
obj_5_results_df.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\objective_5_results.csv", 
    index = False
)

___

Finally we should inspect the F1 overall changes from original through paraphrases for specifics.

In [39]:
# create table for f1 scores
f1_tab = obj_5_results_df.pivot(
    index = "Model",
    columns = "Condition",
    values = "F1"
)

# calculate the change from the original values to paraphrases
f1_tab["Light change"] = (f1_tab["Light"] - f1_tab["Original"])
f1_tab["Translation change"] = (f1_tab["Translation"] - f1_tab["Original"])
f1_tab["Heavy change"] = (f1_tab["Heavy"] - f1_tab["Original"])

# specify column order
f1_tab = f1_tab[
    [
        "Original", "Light", "Translation", "Heavy",
        "Light change", "Translation change", "Heavy change"
    ]
]

f1_tab = f1_tab.reset_index()
# drop condition header
f1_tab.columns.name = None
# round to 3dp
f1_tab = f1_tab.round(3)
# hiding index tidies up the 2 seperate headers
display(f1_tab.style.hide(axis = "index"))

Model,Original,Light,Translation,Heavy,Light change,Translation change,Heavy change
All features,1.000000,0.971000,0.983000,0.789000,-0.029000,-0.017000,-0.211000
Robust features,0.919000,0.895000,0.899000,0.818000,-0.023000,-0.019000,-0.100000


And now we can convert these changes to percentages to observe real change as both models did not start at the same point (original scores were 1 vs 0.919).

In [40]:
# lopp through conditions
for condition in ["Light", "Translation", "Heavy"]:

    # and for each condition calc the % change from the original value
    f1_tab[f"{condition} % change"] = (
        (f1_tab[condition] - f1_tab["Original"])
        / f1_tab["Original"]) * 100

# specify column order, no need for original value columns in this one
f1_tab = f1_tab[["Model", "Light % change", "Translation % change", "Heavy % change"]]

# all below necessary just to get table displaying correctly
f1_tab = f1_tab.round(3)
display(
    f1_tab.style
    .format("{:.3f}", subset = ["Light % change", "Translation % change", "Heavy % change"])
    .hide(axis = "index")
)

Model,Light % change,Translation % change,Heavy % change
All features,-2.900,-1.700,-21.100
Robust features,-2.612,-2.176,-10.990
